In [3]:
!pip install transformers datasets accelerate scikit-learn pandas matplotlib seaborn -q

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

KeyboardInterrupt: 

In [ ]:
import torch
print(f"GPU 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"메모리: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️ GPU가 없습니다! 런타임 → 런타임 유형 변경 → T4 GPU 선택해주세요")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 작업 폴더 생성
!mkdir -p /content/drive/MyDrive/심화캡스톤모델테스트/experiment_results
print("Drive 연결 완료!")

In [ ]:
"""
=============================================================
이거 진짜냐 — 한국어 NLP 모델 비교 실험
=============================================================
Google Colab에서 실행 (Runtime → Change runtime type → T4 GPU)

실험 목적:
  한국어 뉴스 편향/감정/사실-의견 분류에 최적인 모델을
  동일 조건 비교 실험을 통해 선정한다.

비교 모델 (5개):
  1. klue/roberta-base     — KLUE 벤치마크 기준 모델
  2. klue/roberta-large    — base의 대형 버전
  3. beomi/KcBERT-base     — 한국어 댓글 특화
  4. monologg/koelectra-base-v3-discriminator — 효율적 학습
  5. beomi/KcELECTRA-base  — 댓글 + Electra 구조

평가 지표:
  - Accuracy, F1 (weighted), Precision, Recall
  - 추론 속도 (ms/문장)
  - GPU 메모리 사용량 (MB)

작성: 최재원 (202111580)
=============================================================
"""

# ============================================================
# 0. 환경 설치 (Colab 첫 셀에서 실행)
# ============================================================
# !pip install transformers datasets accelerate scikit-learn pandas matplotlib seaborn -q

import os
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from datetime import datetime
from pathlib import Path

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

# 한글 폰트 (Colab 환경)
plt.rcParams["font.family"] = "NanumGothic"
plt.rcParams["axes.unicode_minus"] = False


# ============================================================
# 1. 설정
# ============================================================
class Config:
    """실험 설정 — 필요에 따라 수정"""

    # 비교할 모델 목록
    MODELS = [
        "klue/roberta-base",
        "klue/roberta-large",
        "beomi/KcBERT-base",
        "monologg/koelectra-base-v3-discriminator",
        "beomi/KcELECTRA-base",
    ]

    # 태스크 설정
    TASK = "sentiment"  # "sentiment" | "fact_opinion" | "bias"
    NUM_LABELS = 2
    LABEL_NAMES = ["negative", "positive"]

    # 학습 하이퍼파라미터 (모든 모델 동일 조건)
    MAX_LENGTH = 256
    BATCH_SIZE = 16          # Colab T4 기준 (large 모델은 8로 자동 조정)
    LEARNING_RATE = 2e-5
    NUM_EPOCHS = 5
    WARMUP_RATIO = 0.1
    WEIGHT_DECAY = 0.01
    SEED = 42

    # 데이터
    TEST_SIZE = 0.2
    VAL_SIZE = 0.1  # train에서 추가 분리

    # 추론 속도 테스트 샘플 수
    SPEED_TEST_N = 200

    # 출력 디렉토리
    OUTPUT_DIR = "./experiment_results"


cfg = Config()
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# 재현성
torch.manual_seed(cfg.SEED)
np.random.seed(cfg.SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


# ============================================================
# 2. 데이터 로드
# ============================================================
def load_sample_data():
    import os
    if not os.path.exists("nsmc"):
        os.system("git clone https://github.com/e9t/nsmc.git")

    df = pd.read_csv("nsmc/ratings_train.txt", sep="\t")
    df = df.dropna(subset=["document"])
    df = df.rename(columns={"document": "text"})
    df = df[["text", "label"]]

    # 10,000건 샘플링 (클래스당 5,000건)
    df = df.groupby("label").apply(
        lambda x: x.sample(n=5000, random_state=42)
    ).reset_index(drop=True)

    print(f"NSMC 데이터: {len(df)}건")
    print(f"클래스 분포:\n{df['label'].value_counts().sort_index()}")
    return df


def prepare_datasets(df):
    """Train/Val/Test 분리"""
    train_df, test_df = train_test_split(
        df, test_size=cfg.TEST_SIZE, random_state=cfg.SEED, stratify=df["label"]
    )
    train_df, val_df = train_test_split(
        train_df, test_size=cfg.VAL_SIZE, random_state=cfg.SEED, stratify=train_df["label"]
    )

    print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

    return (
        Dataset.from_pandas(train_df.reset_index(drop=True)),
        Dataset.from_pandas(val_df.reset_index(drop=True)),
        Dataset.from_pandas(test_df.reset_index(drop=True)),
    )


# ============================================================
# 3. 학습 + 평가 함수
# ============================================================
def compute_metrics(eval_pred):
    """평가 지표 계산"""
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, average="weighted"),
        "precision": precision_score(labels, preds, average="weighted", zero_division=0),
        "recall": recall_score(labels, preds, average="weighted", zero_division=0),
    }


def run_experiment(model_name, train_dataset, val_dataset, test_dataset):
    """단일 모델 실험 실행"""
    print(f"\n{'='*60}")
    print(f"실험 시작: {model_name}")
    print(f"{'='*60}")

    result = {"model": model_name}

    # ----- 토크나이저 + 모델 로드 -----
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=cfg.NUM_LABELS
    ).to(device)

    # 파라미터 수
    total_params = sum(p.numel() for p in model.parameters()) / 1e6
    result["params_M"] = round(total_params, 1)
    print(f"  파라미터: {total_params:.1f}M")

    # ----- 토큰화 -----
    def tokenize(batch):
        return tokenizer(
            batch["text"],
            padding="max_length",
            truncation=True,
            max_length=cfg.MAX_LENGTH,
        )

    train_tok = train_dataset.map(tokenize, batched=True)
    val_tok = val_dataset.map(tokenize, batched=True)
    test_tok = test_dataset.map(tokenize, batched=True)

    for ds in [train_tok, val_tok, test_tok]:
        ds.set_format("torch", columns=["input_ids", "attention_mask", "label"])

    # ----- 배치 사이즈 조정 (large 모델) -----
    batch_size = cfg.BATCH_SIZE
    if "large" in model_name.lower():
        batch_size = max(4, batch_size // 2)
        print(f"  Large 모델 → 배치 사이즈 {batch_size}로 조정")

    # ----- 학습 -----
    output_dir = os.path.join(cfg.OUTPUT_DIR, model_name.replace("/", "_"))

    training_args = TrainingArguments(
        output_dir=output_dir,
        num_train_epochs=cfg.NUM_EPOCHS,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size * 2,
        learning_rate=cfg.LEARNING_RATE,
        warmup_ratio=cfg.WARMUP_RATIO,
        weight_decay=cfg.WEIGHT_DECAY,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1",
        greater_is_better=True,
        logging_steps=20,
        seed=cfg.SEED,
        fp16=torch.cuda.is_available(),  # GPU 있으면 mixed precision
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_tok,
        eval_dataset=val_tok,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    )

    # 학습 시간 측정
    train_start = time.time()
    trainer.train()
    train_time = time.time() - train_start
    result["train_time_sec"] = round(train_time, 1)
    print(f"  학습 시간: {train_time:.1f}초")

    # ----- 테스트셋 평가 -----
    eval_result = trainer.evaluate(test_tok)
    result["accuracy"] = round(eval_result["eval_accuracy"] * 100, 2)
    result["f1"] = round(eval_result["eval_f1"] * 100, 2)
    result["precision"] = round(eval_result["eval_precision"] * 100, 2)
    result["recall"] = round(eval_result["eval_recall"] * 100, 2)

    print(f"  Accuracy: {result['accuracy']}%")
    print(f"  F1 Score: {result['f1']}%")

    # ----- 상세 분류 리포트 -----
    preds_output = trainer.predict(test_tok)
    preds = np.argmax(preds_output.predictions, axis=-1)
    labels = preds_output.label_ids

    report = classification_report(
        labels, preds,
        target_names=cfg.LABEL_NAMES,
        output_dict=True,
    )
    result["classification_report"] = report

    # Confusion Matrix
    cm = confusion_matrix(labels, preds)
    result["confusion_matrix"] = cm.tolist()

    # ----- 추론 속도 측정 -----
    model.eval()
    test_texts = [test_dataset[i]["text"] for i in range(min(cfg.SPEED_TEST_N, len(test_dataset)))]

    torch.cuda.synchronize() if torch.cuda.is_available() else None
    speed_start = time.time()

    with torch.no_grad():
        for text in test_texts:
            inputs = tokenizer(
                text, return_tensors="pt",
                truncation=True, max_length=cfg.MAX_LENGTH, padding=True,
            ).to(device)
            _ = model(**inputs)

    torch.cuda.synchronize() if torch.cuda.is_available() else None
    speed_time = time.time() - speed_start

    result["inference_ms"] = round((speed_time / len(test_texts)) * 1000, 2)
    print(f"  추론 속도: {result['inference_ms']}ms / 문장")

    # ----- GPU 메모리 측정 -----
    if torch.cuda.is_available():
        result["gpu_memory_MB"] = round(torch.cuda.max_memory_allocated() / 1e6, 1)
        torch.cuda.reset_peak_memory_stats()
    else:
        result["gpu_memory_MB"] = 0

    # 메모리 정리
    del model, trainer
    torch.cuda.empty_cache() if torch.cuda.is_available() else None

    return result


# ============================================================
# 4. 결과 시각화
# ============================================================
def visualize_results(results_df):
    """실험 결과 시각화"""

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle("News Lens — 한국어 NLP 모델 비교 실험 결과", fontsize=16, fontweight="bold")

    models_short = [m.split("/")[-1] for m in results_df["model"]]
    colors = ["#3B82F6", "#0EA5E9", "#10B981", "#F59E0B", "#8B5CF6"]

    # 1. Accuracy 비교
    ax = axes[0][0]
    bars = ax.bar(models_short, results_df["accuracy"], color=colors)
    ax.set_title("Accuracy (%)", fontweight="bold")
    ax.set_ylim(max(0, results_df["accuracy"].min() - 10), 100)
    for bar, val in zip(bars, results_df["accuracy"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{val:.1f}", ha="center", va="bottom", fontsize=9)
    ax.tick_params(axis="x", rotation=25)

    # 2. F1 Score 비교
    ax = axes[0][1]
    bars = ax.bar(models_short, results_df["f1"], color=colors)
    ax.set_title("F1 Score (weighted, %)", fontweight="bold")
    ax.set_ylim(max(0, results_df["f1"].min() - 10), 100)
    for bar, val in zip(bars, results_df["f1"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{val:.1f}", ha="center", va="bottom", fontsize=9)
    ax.tick_params(axis="x", rotation=25)

    # 3. 추론 속도 비교
    ax = axes[0][2]
    bars = ax.bar(models_short, results_df["inference_ms"], color=colors)
    ax.set_title("추론 속도 (ms/문장, 낮을수록 좋음)", fontweight="bold")
    for bar, val in zip(bars, results_df["inference_ms"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                f"{val:.1f}", ha="center", va="bottom", fontsize=9)
    ax.tick_params(axis="x", rotation=25)

    # 4. 파라미터 수
    ax = axes[1][0]
    bars = ax.bar(models_short, results_df["params_M"], color=colors)
    ax.set_title("파라미터 수 (M)", fontweight="bold")
    for bar, val in zip(bars, results_df["params_M"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f"{val:.0f}M", ha="center", va="bottom", fontsize=9)
    ax.tick_params(axis="x", rotation=25)

    # 5. GPU 메모리
    ax = axes[1][1]
    bars = ax.bar(models_short, results_df["gpu_memory_MB"], color=colors)
    ax.set_title("GPU 메모리 사용량 (MB)", fontweight="bold")
    for bar, val in zip(bars, results_df["gpu_memory_MB"]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f"{val:.0f}", ha="center", va="bottom", fontsize=9)
    ax.tick_params(axis="x", rotation=25)

    # 6. F1 vs 추론 속도 산점도 (trade-off)
    ax = axes[1][2]
    for i, (model, row) in enumerate(results_df.iterrows()):
        ax.scatter(row["inference_ms"], row["f1"],
                   s=row["params_M"] * 2, c=colors[i], alpha=0.7,
                   edgecolors="black", linewidth=0.5)
        ax.annotate(models_short[i],
                    (row["inference_ms"], row["f1"]),
                    textcoords="offset points", xytext=(5, 5), fontsize=8)
    ax.set_xlabel("추론 속도 (ms) →")
    ax.set_ylabel("F1 Score (%) ↑")
    ax.set_title("성능 vs 속도 Trade-off\n(원 크기 = 파라미터 수)", fontweight="bold")

    plt.tight_layout()
    plt.savefig(os.path.join(cfg.OUTPUT_DIR, "model_comparison.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print(f"차트 저장: {cfg.OUTPUT_DIR}/model_comparison.png")


def print_summary_table(results_df):
    """최종 비교 요약 테이블 출력"""

    print("\n" + "=" * 80)
    print("📊 최종 모델 비교 요약")
    print("=" * 80)

    summary = results_df[[
        "model", "accuracy", "f1", "precision", "recall",
        "inference_ms", "params_M", "gpu_memory_MB", "train_time_sec"
    ]].copy()

    summary["model"] = summary["model"].apply(lambda x: x.split("/")[-1])
    summary = summary.rename(columns={
        "model": "모델",
        "accuracy": "Acc(%)",
        "f1": "F1(%)",
        "precision": "Prec(%)",
        "recall": "Rec(%)",
        "inference_ms": "속도(ms)",
        "params_M": "파라미터(M)",
        "gpu_memory_MB": "GPU(MB)",
        "train_time_sec": "학습(초)",
    })

    print(summary.to_string(index=False))

    # 최적 모델 추천
    best_f1 = results_df.loc[results_df["f1"].idxmax()]
    best_speed = results_df.loc[results_df["inference_ms"].idxmin()]
    best_balance = results_df.loc[
        (results_df["f1"] / results_df["f1"].max() * 0.6 +
         results_df["inference_ms"].min() / results_df["inference_ms"] * 0.4).idxmax()
    ]

    print(f"\n🏆 F1 최고: {best_f1['model']} ({best_f1['f1']}%)")
    print(f"⚡ 속도 최고: {best_speed['model']} ({best_speed['inference_ms']}ms)")
    print(f"⚖️  균형 최적: {best_balance['model']} (F1={best_balance['f1']}%, 속도={best_balance['inference_ms']}ms)")

    print("\n💡 추천:")
    print(f"  뉴스 본문 분석 → {best_balance['model']} (성능/효율 균형)")
    if best_f1["model"] != best_balance["model"]:
        print(f"  정밀 분석 필요 시 → {best_f1['model']} (F1 최고)")


def save_results(results):
    """결과 JSON 저장"""
    # confusion_matrix를 serializable하게 변환
    for r in results:
        if "confusion_matrix" in r:
            r["confusion_matrix"] = [list(row) for row in r["confusion_matrix"]]
        if "classification_report" in r:
            # numpy float → python float
            report = r["classification_report"]
            for k, v in report.items():
                if isinstance(v, dict):
                    report[k] = {kk: float(vv) if isinstance(vv, (np.floating, float)) else vv for kk, vv in v.items()}
                elif isinstance(v, (np.floating, float)):
                    report[k] = float(v)

    filepath = os.path.join(cfg.OUTPUT_DIR, "experiment_results.json")
    with open(filepath, "w", encoding="utf-8") as f:
        json.dump({
            "experiment_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
            "config": {
                "task": cfg.TASK,
                "num_labels": cfg.NUM_LABELS,
                "max_length": cfg.MAX_LENGTH,
                "batch_size": cfg.BATCH_SIZE,
                "learning_rate": cfg.LEARNING_RATE,
                "num_epochs": cfg.NUM_EPOCHS,
                "seed": cfg.SEED,
            },
            "results": results,
        }, f, ensure_ascii=False, indent=2)

    print(f"\n결과 저장: {filepath}")


# ============================================================
# 5. 메인 실행
# ============================================================
def main():
    print("=" * 60)
    print("News Lens — 한국어 NLP 모델 비교 실험")
    print(f"태스크: {cfg.TASK} | 모델 수: {len(cfg.MODELS)}")
    print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
    print("=" * 60)

    # 데이터 로드
    df = load_sample_data()
    train_dataset, val_dataset, test_dataset = prepare_datasets(df)

    # 모델별 실험 실행
    all_results = []

    for model_name in cfg.MODELS:
        try:
            result = run_experiment(model_name, train_dataset, val_dataset, test_dataset)
            all_results.append(result)
        except Exception as e:
            print(f"\n❌ {model_name} 실험 실패: {e}")
            all_results.append({
                "model": model_name,
                "error": str(e),
                "accuracy": 0, "f1": 0, "precision": 0, "recall": 0,
                "inference_ms": 999, "params_M": 0, "gpu_memory_MB": 0,
                "train_time_sec": 0,
            })

    # 결과 정리
    results_df = pd.DataFrame(all_results)

    # 시각화
    visualize_results(results_df)

    # 요약 테이블
    print_summary_table(results_df)

    # JSON 저장
    save_results(all_results)

    # CSV 저장
    csv_path = os.path.join(cfg.OUTPUT_DIR, "model_comparison.csv")
    results_df[[
        "model", "accuracy", "f1", "precision", "recall",
        "inference_ms", "params_M", "gpu_memory_MB", "train_time_sec"
    ]].to_csv(csv_path, index=False, encoding="utf-8-sig")
    print(f"CSV 저장: {csv_path}")

    return results_df


# ============ 맨 아래 이 부분을 교체 ============
# 결과를 Drive에 저장하도록 경로 변경
cfg.OUTPUT_DIR = "/content/drive/MyDrive/심화캡스톤모델테스트/experiment_results_nsmc"
import os
os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)

# 실험 실행
results = main()

In [ ]:
!pip uninstall transformers torch accelerate -y
!pip install transformers torch accelerate scikit-learn pandas matplotlib seaborn datasets -q

In [ ]:
from IPython.display import Image, display
display(Image("/content/drive/MyDrive/심화캡스톤모델테스트/experiment_results/model_comparison.png"))

In [ ]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/심화캡스톤모델테스트/experiment_results/model_comparison.csv")
df

In [ ]:
import json
with open("/content/drive/MyDrive/심화캡스톤모델테스트/experiment_results/experiment_results.json", "r") as f:
    data = json.load(f)

print(f"실험 일시: {data['experiment_date']}")
print(f"실험 조건: {json.dumps(data['config'], indent=2)}")

for r in data['results']:
    print(f"\n{'='*50}")
    print(f"모델: {r['model']}")
    print(f"  Accuracy: {r['accuracy']}%")
    print(f"  F1 Score: {r['f1']}%")
    print(f"  추론 속도: {r['inference_ms']}ms")
    print(f"  GPU 메모리: {r['gpu_memory_MB']}MB")


In [ ]:
# NSMC 데이터 다운로드
!git clone https://github.com/e9t/nsmc.git

# load_sample_data() 함수를 이걸로 교체
def load_sample_data():
    df = pd.read_csv("nsmc/ratings_train.txt", sep="\t")
    df = df.dropna(subset=["document"])
    df = df.rename(columns={"document": "text"})
    df = df[["text", "label"]]
    # 10,000건 샘플링 (시간 절약)
    df = df.groupby("label").apply(
        lambda x: x.sample(n=5000, random_state=42)
    ).reset_index(drop=True)
    print(f"NSMC 데이터: {len(df)}건")
    return df